# 📄 Universal Document Restoration Tool

Turns **any** scanned, aged, stained or damaged document image into a clean,
typeset PDF — exactly as if it had been retyped from scratch on fresh paper.

### How it works
1. Point it at any image file (JPEG, PNG, TIFF, BMP, WebP)
2. Claude Vision reads every word, number, table, and handwritten note
3. ReportLab typesets the result into a brand-new clean PDF

### Requirements
- Python 3.9+
- An Anthropic API key — get one at [console.anthropic.com](https://console.anthropic.com)
- Run **Step 1** once to install packages, then **Steps 2–6** for each document

## Step 1 — Install packages  *(run once)*

In [ ]:
!pip install anthropic reportlab Pillow tqdm

## Step 2 — Imports

In [2]:
import os, sys, base64, io, textwrap
from pathlib import Path

import anthropic
from PIL import Image
from reportlab.lib.pagesizes import A4, letter, landscape
from reportlab.lib.units import mm
from reportlab.lib import colors
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Table, TableStyle, HRFlowable, PageBreak
)
from tqdm import tqdm
from IPython.display import IFrame, display, HTML

print('✅  All imports OK')

✅  All imports OK


## Step 3 — Configuration

**Edit these three lines** for each document you want to restore.

In [ ]:
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  EDIT THESE                                                             │
# └─────────────────────────────────────────────────────────────────────────┘

INPUT_IMAGE  = ''   # ← any JPEG / PNG / TIFF / BMP / WebP
OUTPUT_PDF   = ''   # ← where the clean PDF is written
API_KEY      = ''   # ← paste key here, OR set env var ANTHROPIC_API_KEY

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  OPTIONAL TUNING  (safe to leave as-is)                                 │
# └─────────────────────────────────────────────────────────────────────────┘

PAGE_SIZE     = A4          # A4  |  letter  |  landscape(A4)  |  landscape(letter)
FONT_SIZE     = 8           # body text size in points (7–11 works well)
MARGINS_MM    = 12          # page margins in mm
MODEL         = 'claude-sonnet-4-6'   # claude-sonnet-4-6 (fast) | claude-opus-4-5 (best)

# Optional: tell Claude what kind of document this is — improves accuracy
# for specialist jargon, abbreviations, and unusual layouts.
# Leave as '' for fully automatic detection.
DOCUMENT_HINT = ''
# Examples:
#   DOCUMENT_HINT = 'Meteorological observation logbook, 1950s, British West Africa.'
#   DOCUMENT_HINT = 'Handwritten financial ledger, Victorian era.'
#   DOCUMENT_HINT = 'Medical patient record with physician handwriting.'
#   DOCUMENT_HINT = 'Legal contract with witness signatures.'

# ── resolve API key ────────────────────────────────────────────────────────
if not API_KEY:
    API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
if not API_KEY:
    API_KEY = input(INSERT API KEY).strip()

client = anthropic.Anthropic(api_key=API_KEY)
print(f'✅  Config OK')
print(f'   Input : {INPUT_IMAGE}')
print(f'   Output: {OUTPUT_PDF}')
print(f'   Model : {MODEL}')

## Step 4 — Core functions

Run this cell once. It defines `ocr_document()` and `build_pdf()` — the two
functions that do all the work.

In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
#  PART A — OCR  (Claude Vision reads the document)
# ═══════════════════════════════════════════════════════════════════════════════

_SUPPORTED = {'.jpg', '.jpeg', '.png', '.gif', '.webp', '.tif', '.tiff', '.bmp'}

def _load_image(path: str) -> tuple:
    """
    Return (base64_string, media_type) for any image file.
    Converts TIFF / BMP / unsupported formats to JPEG automatically.
    """
    p      = Path(path)
    suffix = p.suffix.lower()
    native = {'.jpg': 'image/jpeg', '.jpeg': 'image/jpeg',
              '.png': 'image/png',  '.gif':  'image/gif',
              '.webp': 'image/webp'}
    if suffix in native:
        raw = p.read_bytes()
        return base64.standard_b64encode(raw).decode(), native[suffix]
    # Convert anything else to JPEG
    buf = io.BytesIO()
    Image.open(path).convert('RGB').save(buf, format='JPEG', quality=95)
    return base64.standard_b64encode(buf.getvalue()).decode(), 'image/jpeg'


_OCR_PROMPT = """\
You are an expert document transcription assistant. Your job is to produce a
complete, faithful transcription of every piece of text visible in this scanned
document image — printed AND handwritten — so that a program can typeset the
content into a clean new PDF.

OUTPUT FORMAT
=============
Use ONLY the following prefixed line types. One item per line.

  HEADING:       <primary title / document name>
  SUBHEADING:    <secondary title or subtitle>
  SECTION:       <section divider label, e.g. "EXPOSURE OF INSTRUMENTS">
  FIELD:         <label> | <value>
  TABLE_HEADER:  <col1> | <col2> | <col3> | ...
  TABLE_ROW:     <val1> | <val2> | <val3> | ...
  TEXT:          <free-form paragraph or sentence>
  NOTE:          <footnote, marginal note, or small-print remark>

RULES
=====
1. Transcribe EVERYTHING visible — no skipping, no summarising.
2. Preserve ALL columns in tables. Use a single space for empty cells:  |   |
3. For illegible text write [illegible] in that position.
4. Follow reading order: top-to-bottom, left-to-right.
5. When a document has two pages side by side, transcribe the LEFT page
   completely first (top to bottom), then the RIGHT page.
6. Output ONLY the transcription — zero commentary, explanation or apology.
"""


def ocr_document(image_path: str,
                 hint: str = '',
                 model: str = 'claude-sonnet-4-6',
                 max_tokens: int = 8192) -> str:
    """
    Send a document image to Claude Vision and return the structured transcription.

    Parameters
    ----------
    image_path  : path to any image file (JPEG/PNG/TIFF/BMP/WebP)
    hint        : optional one-sentence description of the document type
    model       : Anthropic model ID
    max_tokens  : maximum output tokens (increase for very long documents)

    Returns
    -------
    str  — structured transcription ready for build_pdf()
    """
    print(f'📤  Reading "{Path(image_path).name}" with Claude Vision...')
    b64, media_type = _load_image(image_path)

    prompt = _OCR_PROMPT
    if hint:
        prompt += f'\nDocument context: {hint}\n'

    resp = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[{'role': 'user', 'content': [
            {'type': 'image',
             'source': {'type': 'base64',
                        'media_type': media_type,
                        'data': b64}},
            {'type': 'text', 'text': prompt}
        ]}]
    )

    text = resp.content[0].text
    print(f'✅  Transcription received  '
          f'({len(text):,} chars  |  '
          f'in {resp.usage.input_tokens:,} tok  '
          f'out {resp.usage.output_tokens:,} tok)')
    return text


# ═══════════════════════════════════════════════════════════════════════════════
#  PART B — PDF TYPESETTER  (ReportLab renders the transcription)
# ═══════════════════════════════════════════════════════════════════════════════

def build_pdf(transcription: str,
              output_path: str,
              page_size=A4,
              font_size: int = 8,
              margins_mm: int = 12) -> str:
    """
    Parse a structured transcription and render it as a clean typeset PDF.

    Recognised prefixes
    -------------------
    HEADING      → centred bold large title
    SUBHEADING   → centred bold medium title
    SECTION      → centred bold with horizontal rules above and below
    FIELD        → bold label + value on one line
    TABLE_HEADER → shaded bold header row in a grid table
    TABLE_ROW    → data row in the same table
    TEXT         → normal paragraph
    NOTE         → italic small-print footnote

    Parameters
    ----------
    transcription : output of ocr_document()
    output_path   : path for the output PDF file
    page_size     : reportlab page size (A4, letter, landscape(A4), …)
    font_size     : base font size in points
    margins_mm    : page margins in mm

    Returns
    -------
    str  — path to the saved PDF
    """
    m  = margins_mm * mm
    pw = page_size[0] - 2 * m   # usable page width

    doc = SimpleDocTemplate(
        output_path, pagesize=page_size,
        leftMargin=m, rightMargin=m,
        topMargin=m,  bottomMargin=m,
    )

    # ── Paragraph styles ──────────────────────────────────────────────────────
    S = {
        'H1': ParagraphStyle('H1',
            fontName='Helvetica-Bold', fontSize=font_size+5,
            spaceAfter=5, spaceBefore=8, alignment=1),          # centred
        'H2': ParagraphStyle('H2',
            fontName='Helvetica-Bold', fontSize=font_size+3,
            spaceAfter=4, spaceBefore=6, alignment=1),
        'SEC': ParagraphStyle('SEC',
            fontName='Helvetica-Bold', fontSize=font_size+1,
            spaceAfter=2, spaceBefore=2, alignment=1),
        'FLD': ParagraphStyle('FLD',
            fontName='Helvetica', fontSize=font_size,
            spaceAfter=1, spaceBefore=1, leading=font_size+3),
        'TXT': ParagraphStyle('TXT',
            fontName='Helvetica', fontSize=font_size,
            spaceAfter=3, spaceBefore=2, leading=font_size+3),
        'NOTE': ParagraphStyle('NOTE',
            fontName='Helvetica-Oblique', fontSize=font_size-1,
            spaceAfter=2, spaceBefore=2, leading=font_size+1,
            textColor=colors.HexColor('#444444')),
        'TH': ParagraphStyle('TH',
            fontName='Helvetica-Bold', fontSize=font_size-1, alignment=1),
        'TD': ParagraphStyle('TD',
            fontName='Helvetica',      fontSize=font_size-1, alignment=1),
    }

    # ── State ─────────────────────────────────────────────────────────────────
    story    = []
    tbl_rows = []   # accumulated (kind, [cells])

    def flush():
        """Emit any buffered table rows."""
        if not tbl_rows:
            return
        ncols = max(len(r) for _, r in tbl_rows)
        data, hdr_idx = [], []
        for i, (kind, cells) in enumerate(tbl_rows):
            padded = (cells + [''] * ncols)[:ncols]
            sty    = S['TH'] if kind == 'h' else S['TD']
            data.append([Paragraph(_esc(c), sty) for c in padded])
            if kind == 'h':
                hdr_idx.append(i)

        cw  = pw / ncols
        tbl = Table(data, colWidths=[cw]*ncols, repeatRows=1)
        ts  = TableStyle([
            ('GRID',          (0,0), (-1,-1), 0.4,  colors.black),
            ('FONTSIZE',      (0,0), (-1,-1), font_size-1),
            ('TOPPADDING',    (0,0), (-1,-1), 2),
            ('BOTTOMPADDING', (0,0), (-1,-1), 2),
            ('LEFTPADDING',   (0,0), (-1,-1), 3),
            ('RIGHTPADDING',  (0,0), (-1,-1), 3),
            ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
            ('ALIGN',         (0,0), (-1,-1), 'CENTER'),
        ])
        for i in hdr_idx:
            ts.add('BACKGROUND', (0,i), (-1,i), colors.HexColor('#D5D5D5'))
            ts.add('FONTNAME',   (0,i), (-1,i), 'Helvetica-Bold')
        tbl.setStyle(ts)
        story.append(tbl)
        story.append(Spacer(1, 4))
        tbl_rows.clear()

    def _esc(s):
        """Escape ReportLab XML special characters."""
        return (str(s)
                .replace('&', '&amp;')
                .replace('<', '&lt;')
                .replace('>', '&gt;'))

    def pipe(s):
        return [c.strip() for c in s.split('|')]

    # ── Parse transcription line by line ──────────────────────────────────────
    for raw in transcription.splitlines():
        line = raw.strip()
        if not line:
            continue

        if ':' not in line:
            flush()
            story.append(Paragraph(_esc(line), S['TXT']))
            continue

        pre, _, rest = line.partition(':')
        key  = pre.strip().upper()
        rest = rest.strip()

        if key == 'HEADING':
            flush()
            story.append(Paragraph(_esc(rest), S['H1']))

        elif key == 'SUBHEADING':
            flush()
            story.append(Paragraph(_esc(rest), S['H2']))

        elif key == 'SECTION':
            flush()
            story.append(Spacer(1, 6))
            story.append(HRFlowable(width='100%', thickness=0.7,
                                    color=colors.black))
            story.append(Paragraph(_esc(rest), S['SEC']))
            story.append(HRFlowable(width='100%', thickness=0.7,
                                    color=colors.black))
            story.append(Spacer(1, 3))

        elif key == 'FIELD':
            flush()
            parts = pipe(rest)
            if len(parts) >= 2:
                lbl = _esc(parts[0])
                val = _esc('   '.join(parts[1:]))
                story.append(Paragraph(f'<b>{lbl}:</b>  {val}', S['FLD']))
            else:
                story.append(Paragraph(_esc(rest), S['FLD']))

        elif key == 'TABLE_HEADER':
            tbl_rows.append(('h', pipe(rest)))

        elif key == 'TABLE_ROW':
            tbl_rows.append(('r', pipe(rest)))

        elif key == 'NOTE':
            flush()
            story.append(Paragraph('* ' + _esc(rest), S['NOTE']))

        elif key == 'TEXT':
            flush()
            story.append(Paragraph(_esc(rest), S['TXT']))

        else:
            # Unknown prefix — render as plain text
            flush()
            story.append(Paragraph(_esc(line), S['TXT']))

    flush()   # emit any trailing table

    doc.build(story)
    print(f'✅  Clean PDF saved  →  {output_path}')
    return output_path


# ═══════════════════════════════════════════════════════════════════════════════
#  PART C — ONE-SHOT WRAPPER
# ═══════════════════════════════════════════════════════════════════════════════

def restore_document(image_path: str,
                     output_pdf: str   = None,
                     hint: str         = '',
                     page_size         = A4,
                     font_size: int    = 8,
                     margins_mm: int   = 12,
                     model: str        = 'claude-sonnet-4-6',
                     save_txt: bool    = False) -> dict:
    """
    Complete pipeline: image  →  OCR  →  clean PDF.

    Parameters
    ----------
    image_path  : input image (any format)
    output_pdf  : output PDF path  (defaults to <stem>_restored.pdf)
    hint        : optional document-type description for better OCR
    page_size   : A4 | letter | landscape(A4) | landscape(letter)
    font_size   : body text size in points
    margins_mm  : page margins in mm
    model       : claude-sonnet-4-6 (fast) | claude-opus-4-5 (highest accuracy)
    save_txt    : if True, also save the raw transcription as a .txt file

    Returns
    -------
    dict  {'pdf': path, 'txt': path_or_None, 'transcription': str}
    """
    stem = Path(image_path).stem
    if output_pdf is None:
        output_pdf = str(Path(image_path).parent / f'{stem}_restored.pdf')

    transcription = ocr_document(image_path, hint=hint, model=model)

    txt_path = None
    if save_txt:
        txt_path = output_pdf.replace('.pdf', '_transcription.txt')
        Path(txt_path).write_text(transcription, encoding='utf-8')
        print(f'📝  Transcription saved  →  {txt_path}')

    build_pdf(transcription, output_pdf,
              page_size=page_size, font_size=font_size,
              margins_mm=margins_mm)

    return {'pdf': output_pdf, 'txt': txt_path, 'transcription': transcription}


# ═══════════════════════════════════════════════════════════════════════════════
#  PART D — BATCH FOLDER
# ═══════════════════════════════════════════════════════════════════════════════

SUPPORTED_EXT = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.webp'}

def restore_folder(input_folder: str,
                   output_folder: str,
                   hint: str       = '',
                   page_size       = A4,
                   font_size: int  = 8,
                   margins_mm: int = 12,
                   model: str      = 'claude-sonnet-4-6',
                   save_txt: bool  = False) -> list:
    """
    Restore every document image in input_folder.
    Writes one clean PDF per image into output_folder.
    Accepts the same keyword arguments as restore_document().
    """
    in_dir  = Path(input_folder)
    out_dir = Path(output_folder)
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(f for f in in_dir.iterdir()
                   if f.suffix.lower() in SUPPORTED_EXT)
    print(f'🗂️   Found {len(files)} image(s) in "{input_folder}"')

    results = []
    for f in tqdm(files, desc='Restoring'):
        out_pdf = str(out_dir / (f.stem + '_restored.pdf'))
        try:
            r = restore_document(str(f), output_pdf=out_pdf, hint=hint,
                                 page_size=page_size, font_size=font_size,
                                 margins_mm=margins_mm, model=model,
                                 save_txt=save_txt)
            results.append({'source': str(f), 'output': out_pdf,
                            'ok': True, **r})
        except Exception as e:
            print(f'  ⚠️  {f.name}: {e}')
            results.append({'source': str(f), 'output': None,
                            'ok': False, 'error': str(e)})

    ok = sum(1 for r in results if r['ok'])
    print(f'\n✅  Done — {ok}/{len(files)} restored  →  "{output_folder}"')
    return results


print('✅  All functions defined (ocr_document, build_pdf, restore_document, restore_folder)')

✅  All functions defined (ocr_document, build_pdf, restore_document, restore_folder)


## Step 5 — Restore a single document

This is the main cell. It reads the image, OCRs it with Claude Vision,
and writes a clean PDF. **Just run it.**

In [ ]:
result = restore_document(
    image_path  = INPUT_IMAGE,
    output_pdf  = OUTPUT_PDF,
    hint        = DOCUMENT_HINT,
    page_size   = PAGE_SIZE,
    font_size   = FONT_SIZE,
    margins_mm  = MARGINS_MM,
    model       = MODEL,
    save_txt    = True,     # also save raw transcription as .txt alongside the PDF
)

print('\n' + '─'*60)
print('TRANSCRIPTION PREVIEW (first 2000 chars):')
print('─'*60)
print(result['transcription'][:2000])

## Step 6 — Fix OCR errors without re-calling the API  *(optional)*

If Claude misread any value, edit the transcription text below and re-build
the PDF. No API call needed — costs nothing.

In [ ]:
# ── Option A: fix a specific value in the transcription ───────────────────────
# corrected = result['transcription'].replace('wrong value', 'correct value')
# build_pdf(corrected, OUTPUT_PDF, page_size=PAGE_SIZE,
#           font_size=FONT_SIZE, margins_mm=MARGINS_MM)

# ── Option B: open the .txt file in any text editor, save, then reload ────────
txt_path = ''
corrected = Path(txt_path).read_text(encoding='utf-8')
build_pdf(corrected, OUTPUT_PDF, page_size=PAGE_SIZE,
          font_size=FONT_SIZE, margins_mm=MARGINS_MM)

print('Uncomment one of the blocks above to rebuild with corrections.')

## Step 7 — Batch restore a folder  *(optional)*

Point it at a folder and it restores every image inside.

In [8]:
# results = restore_folder(
#     input_folder  = 'scans/',          # folder containing your image files
#     output_folder = 'restored/',       # folder to write clean PDFs into
#     hint          = DOCUMENT_HINT,
#     page_size     = PAGE_SIZE,
#     font_size     = FONT_SIZE,
#     margins_mm    = MARGINS_MM,
#     model         = MODEL,
#     save_txt      = True,
# )

print('Uncomment the block above and set your folder paths to run batch mode.')

Uncomment the block above and set your folder paths to run batch mode.


---
## Appendix — Document hints for specialist content

Set `DOCUMENT_HINT` in Step 3 to improve accuracy for specific document types.

| Document type | Suggested hint |
|---|---|
| Meteorological logbook | `'Meteorological observation logbook, 1950s, British West Africa.'` |
| Handwritten ledger | `'Handwritten financial ledger, Victorian era.'` |
| Medical record | `'Medical patient record with handwritten physician notes.'` |
| Legal contract | `'Legal contract with witness signatures and clause numbering.'` |
| Ship's log | `'Ship captain log with weather, position, and course entries.'` |
| Census record | `'Historical census form with handwritten household entries.'` |
| Birth / death certificate | `'Civil registration certificate, 19th century.'` |
| Military record | `'Military service record with ranks, dates and postings.'` |

### Landscape / oversized documents
```python
from reportlab.lib.pagesizes import landscape, A4, A3
PAGE_SIZE = landscape(A4)   # wide page — good for tables with many columns
PAGE_SIZE = A3              # larger paper
```

### Higher accuracy (slower, costs more)
```python
MODEL = 'claude-opus-4-5'   # best for heavily degraded or unusual handwriting
```

### Saving the raw transcription for review
```python
restore_document(..., save_txt=True)   # writes <name>_transcription.txt alongside the PDF
```